# Lab 1.1: Build an Agent-Ready Work Plan

**Day 1 - Session 1**

Goal: Build and machine-check a dependency-aware execution plan for a shipment-notification feature.

> Open this notebook in Google Colab:
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/agent-orchestration-companion/blob/main/day1/lab1_1_agent-ready-work-plan-colab/start/lab1_1_agent-ready-work-plan-colab.ipynb)

In [ ]:
# This lab uses only the Python standard library; no package installation is required.
import sys
print(f"Python {sys.version_info.major}.{sys.version_info.minor} ready; no API key required.")

In [ ]:
import json
from collections import defaultdict

## Section 1: Read the planning evidence

A retail platform needs email and short-message shipment updates. Notifications must use an approved event contract, respect channel preferences, and create an audit record. Authentication, rate limiting, vendor delivery clients, and platform infrastructure are outside scope.

| Path | Purpose | Boundary |
| :--- | :--- | :--- |
| `schemas/shipment-event.json` | Shared event contract | Human-approved structural hub |
| `channels/email.py` | Email rendering | Email task only |
| `channels/sms.py` | Short-message rendering | SMS task only |
| `preferences/reader.py` | Preference lookup | Preference task only |
| `audit/recorder.py` | Audit records | Audit task only |
| `router.py` | Coordinates notification paths | One integration owner |
| `tests/test_notifications.py` | Integration acceptance | Integration owner only |

In [ ]:
REQUIRED_FIELDS = {
    "id", "objective", "deliverable", "files", "depends_on",
    "classification", "acceptance", "non_goals",
}
ALLOWED_CLASSIFICATIONS = {"parallel", "sequential", "human-controlled"}
ALLOWED_COMMANDS = {
    "python3 -m unittest tests.test_schema",
    "python3 -m unittest tests.test_email",
    "python3 -m unittest tests.test_sms",
    "python3 -m unittest tests.test_preferences",
    "python3 -m unittest tests.test_audit",
    "python3 -m unittest tests.test_notifications",
    "python3 -m unittest tests.test_security",
}
print(f"Planning contract loaded: {len(REQUIRED_FIELDS)} fields, {len(ALLOWED_COMMANDS)} supplied checks.")

## Section 2: Load the machine-checking contract

The validator checks completeness, unique ownership, supplied commands, reachable dependencies, the schema gate, and the integration owner. It computes execution waves from the dependencies rather than trusting a hand-written schedule.

In [ ]:
def contains_todo(value):
    if isinstance(value, str):
        return "todo" in value.lower()
    if isinstance(value, list):
        return any(contains_todo(item) for item in value)
    if isinstance(value, dict):
        return any(contains_todo(item) for item in value.values())
    return False


def execution_waves(tasks):
    remaining = {task["id"]: set(task["depends_on"]) for task in tasks}
    completed = set()
    waves = []
    while remaining:
        ready = sorted(task_id for task_id, deps in remaining.items() if deps <= completed)
        if not ready:
            return None
        waves.append(ready)
        completed.update(ready)
        for task_id in ready:
            del remaining[task_id]
    return waves


def validate_plan(plan):
    errors = []
    tasks = plan.get("tasks", [])
    if len(tasks) < 5:
        errors.append("Define at least five bounded tasks.")
    if not plan.get("human_approval_point") or contains_todo(plan["human_approval_point"]):
        errors.append("Replace the human approval TODO with an observable checkpoint.")

    task_ids = [task.get("id") for task in tasks]
    if len(task_ids) != len(set(task_ids)):
        errors.append("Task IDs must be unique.")

    owners = defaultdict(list)
    for index, task in enumerate(tasks, start=1):
        missing = REQUIRED_FIELDS - task.keys()
        if missing:
            errors.append(f"Task {index} is missing: {', '.join(sorted(missing))}.")
            continue
        if contains_todo(task):
            errors.append(f"Task {task['id']} still contains TODO text.")
        if task["classification"] not in ALLOWED_CLASSIFICATIONS:
            errors.append(f"Task {task['id']} has an invalid classification.")
        if not task["files"]:
            errors.append(f"Task {task['id']} must own at least one file.")
        if not task["acceptance"]:
            errors.append(f"Task {task['id']} needs an acceptance command.")
        for command in task["acceptance"]:
            if command not in ALLOWED_COMMANDS:
                errors.append(f"Task {task['id']} uses an unsupplied command: {command}")
        if not task["non_goals"]:
            errors.append(f"Task {task['id']} needs at least one non-goal.")
        for path in task["files"]:
            owners[path].append(task["id"])
        for dependency in task["depends_on"]:
            if dependency not in task_ids:
                errors.append(f"Task {task['id']} has unknown dependency {dependency}.")

    for path, task_owners in owners.items():
        if len(task_owners) > 1:
            errors.append(f"File {path} has multiple owners: {', '.join(task_owners)}.")

    by_id = {task.get("id"): task for task in tasks}
    schema = by_id.get("schema-contract")
    if not schema or schema.get("classification") != "human-controlled":
        errors.append("schema-contract must be human-controlled.")
    integration = by_id.get("notification-integration")
    if not integration or "router.py" not in integration.get("files", []):
        errors.append("notification-integration must own router.py.")
    if integration and len(integration.get("depends_on", [])) < 3:
        errors.append("notification-integration needs at least three upstream dependencies.")

    waves = execution_waves(tasks) if tasks and None not in task_ids else None
    if waves is None:
        errors.append("Dependencies contain a cycle or an unreachable task.")
    return errors, waves

## Section 3: Inspect the unsafe starter plan

The naive split starts consumers too early, gives two tasks ownership of `router.py`, and omits required outcomes. Run the cell and use the errors as planning evidence.

In [ ]:
unsafe_starter = {
    "scenario": "Retail shipment notifications",
    "human_approval_point": "TODO",
    "tasks": [
        {"id": "schema", "objective": "TODO define the contract gate",
         "deliverable": "schemas/shipment-event.json",
         "files": ["schemas/shipment-event.json"], "depends_on": [],
         "classification": "parallel", "acceptance": [], "non_goals": []},
        {"id": "email", "objective": "TODO", "deliverable": "channels/email.py",
         "files": ["channels/email.py", "router.py"], "depends_on": [],
         "classification": "parallel", "acceptance": [], "non_goals": []},
        {"id": "sms", "objective": "TODO", "deliverable": "channels/sms.py",
         "files": ["channels/sms.py", "router.py"], "depends_on": [],
         "classification": "parallel", "acceptance": [], "non_goals": []},
    ],
}
starter_errors, _ = validate_plan(unsafe_starter)
print("Unsafe starter findings:")
for error in starter_errors:
    print(f"- {error}")

## Section 4: Define the contract gate

Create the upstream schema task. A human architect must approve this shared contract before any consumer task begins.

In [ ]:
schema_task = {
    "id": "schema-contract",
    "objective": "Define and approve the stable shipment-event contract consumed by all notification workers.",
    "deliverable": "A validated shipment-event JSON schema.",
    "files": ["schemas/shipment-event.json"],
    "depends_on": [],
    "classification": "human-controlled",
    "acceptance": ["python3 -m unittest tests.test_schema"],
    "non_goals": ["Do not implement channel rendering or vendor delivery."],
}

## Section 5: Define parallel consumer tasks

Email, SMS, preference, and audit work can run concurrently only after the schema gate. Each task owns one bounded file and uses one supplied acceptance command.

In [ ]:
def build_consumer_task(task_id, objective, deliverable, owned_file, acceptance_command, non_goal):
    return {
        "id": task_id,
        "objective": objective,
        "deliverable": deliverable,
        "files": [owned_file],
        "depends_on": ["schema-contract"],
        "classification": "parallel",
        "acceptance": [acceptance_command],
        "non_goals": [non_goal],
    }


consumer_tasks = [
    build_consumer_task(
        "email-renderer", "Render an email subject and body from an approved shipment event.",
        "A channel-specific email renderer.", "channels/email.py",
        "python3 -m unittest tests.test_email", "Do not edit the router or send a real email.",
    ),
    build_consumer_task(
        "sms-renderer", "Render a length-bounded short message from an approved shipment event.",
        "A channel-specific short-message renderer.", "channels/sms.py",
        "python3 -m unittest tests.test_sms", "Do not edit the router or call a messaging vendor.",
    ),
    build_consumer_task(
        "preference-reader", "Expose saved channel choices needed by notification routing.",
        "A preference lookup with deterministic fallback behavior.", "preferences/reader.py",
        "python3 -m unittest tests.test_preferences", "Do not redesign customer identity or persistence.",
    ),
    build_consumer_task(
        "audit-recorder", "Record the notification decision and outcome without message content.",
        "A structured audit recorder.", "audit/recorder.py",
        "python3 -m unittest tests.test_audit", "Do not change retention policy or logging infrastructure.",
    ),
]

## Section 6: Reserve integration and record approval

The final task waits for all four consumer artifacts. It alone owns the central router and integration test, then runs notification and security checks.

In [ ]:
integration_task = {
    "id": "notification-integration",
    "objective": "Connect approved channel, preference, and audit artifacts through the central router.",
    "deliverable": "An integrated router with complete acceptance evidence.",
    "files": ["router.py", "tests/test_notifications.py"],
    "depends_on": ["email-renderer", "sms-renderer", "preference-reader", "audit-recorder"],
    "classification": "sequential",
    "acceptance": [
        "python3 -m unittest tests.test_notifications",
        "python3 -m unittest tests.test_security",
    ],
    "non_goals": ["Do not alter the event schema or platform gateway configuration."],
}

human_approval_point = "A human architect approves the schema check before consumer tasks enter Wave 2."

## Section 7: Assemble and inspect the plan

Combine the cards, inspect the generated JSON, and treat validator findings as repair guidance rather than weakening the checks.

In [ ]:
plan = {
    "scenario": "Retail shipment notifications",
    "human_approval_point": human_approval_point,
    "tasks": [schema_task, *consumer_tasks, integration_task],
}
print(json.dumps(plan, indent=2))

errors, waves = validate_plan(plan)
print("\nPLAN VALID" if not errors else "\nPLAN INVALID")
for error in errors:
    print(f"- {error}")

## Section 8: Run the acceptance check

The untouched starter fails here by design. Complete every TODO, rerun the changed cells, and verify the computed contract, parallel, and integration waves.

In [ ]:
errors, waves = validate_plan(plan)
assert not errors, "Repair the plan before handoff:\n- " + "\n- ".join(errors)
print("PLAN VALID")
for number, wave in enumerate(waves, start=1):
    print(f"Wave {number}: {', '.join(wave)}")
print(f"Human approval: {plan['human_approval_point']}")

In [ ]:
print("Lab complete.")
print("Takeaway: Parallelize bounded work only after contracts, file ownership, acceptance checks, and human gates are explicit.")